# YOLOv11 Chess Pieces Detection - Kaggle / Colab Training

This notebook supports training YOLOv11m Chess Piece Detection on **both Google Colab and Kaggle**.
- On **Colab**: Google Drive is mounted and all paths are resolved from the Drive project directory.
- On **Kaggle**: `/kaggle/working/` is the writable workspace; outputs are committed automatically via **Save Version**.

> **Run Step 0 first** — it detects the environment and sets global path variables used by every subsequent cell.

## Step 0: Environment Detection & Path Normalization

This cell **must run first**. It:
1. Detects whether we are running on **Google Colab**, **Kaggle**, or a local machine.
2. On **Colab** — mounts Google Drive and sets all paths relative to the Drive project folder
   (`MyDrive/chess_pieces_detection/`), matching the layout used in `cloud_computing/train.ipynb`.
3. On **Kaggle** — uses `/kaggle/working/` as the writable workspace.
   - `/kaggle/input/` is read-only; attach your dataset there via the Kaggle UI **Data** tab.
   - Outputs written to `/kaggle/working/` are committed automatically when you click
     **Save Version → Save & Run All** (access them from the *Output* tab, or add the committed
     output as a Kaggle Dataset to reuse trained weights in future notebooks).
   - For fully automated cross-session persistence, upload results to an external bucket
     (e.g. Google Cloud Storage) using `gcloud` / `boto3`.

In [ ]:
# ============================================================
# Environment Detection & Unified Path Configuration
# ============================================================
# Detects Colab / Kaggle / local runtime and exposes a single
# set of Path variables consumed by every subsequent cell:
#
#   LOCAL_DIR   - writable directory for datasets (annotations + images)
#   DRIVE_DIR   - persistent storage root for datasets (Drive or None)
#   RUNS_DIR    - writable directory where training results are saved
#   INPUT_DIR   - read-only input datasets directory (Kaggle only)
#   RESUME_CKPT - path to checkpoint to load / resume from
#   ENV         - string: 'colab' | 'kaggle' | 'local'
# ============================================================

import os
import sys
from pathlib import Path

# -- Detect environment ----------------------------------------
def _is_colab() -> bool:
    """Return True when running inside Google Colab."""
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

def _is_kaggle() -> bool:
    """Return True when running inside a Kaggle notebook kernel."""
    return os.environ.get("KAGGLE_KERNEL_RUN_TYPE") is not None or \
           Path("/kaggle/working").exists()

if _is_colab():
    ENV = "colab"
elif _is_kaggle():
    ENV = "kaggle"
else:
    ENV = "local"

print(f"Detected environment: {ENV.upper()}")

# -- Colab -----------------------------------------------------
if ENV == "colab":
    from google.colab import drive  # type: ignore
    print("Mounting Google Drive...")
    drive.mount("/content/drive")

    # Root of the project on Drive -- same layout as cloud_computing/train.ipynb
    # MyDrive/chess_pieces_detection/
    #   datasets/   <- DRIVE_DIR (annotations.json, preprocessed_images.zip)
    #   runs/       <- RUNS_DIR  (training outputs, checkpoints)
    _DRIVE_PROJECT = Path("/content/drive/MyDrive/chess_pieces_detection")

    DRIVE_DIR  = _DRIVE_PROJECT / "datasets"   # persistent dataset backup on Drive
    LOCAL_DIR  = Path("/content/datasets")     # fast local SSD workspace
    RUNS_DIR   = str(_DRIVE_PROJECT / "runs")  # save results directly to Drive
    INPUT_DIR  = None                          # not applicable on Colab

    # Pretrained / resume checkpoint stored on Drive
    RESUME_CKPT = str(
        _DRIVE_PROJECT / "runs" / "chess_detection_yolo11m-2" / "weights" / "last.pt"
    )

    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"  Drive project : {_DRIVE_PROJECT}")
    print(f"  LOCAL_DIR     : {LOCAL_DIR}")
    print(f"  DRIVE_DIR     : {DRIVE_DIR}")
    print(f"  RUNS_DIR      : {RUNS_DIR}")

# -- Kaggle ----------------------------------------------------
elif ENV == "kaggle":
    # /kaggle/working/ is the writable workspace (~20 GB).
    # Files written here persist for the interactive session lifetime AND are
    # committed automatically via Save Version -> Save & Run All.
    # After committing you can:
    #   1. Download output files from the Output tab of the committed version.
    #   2. Add the committed output as a Kaggle Dataset to reuse weights
    #      in future notebooks without re-training.
    # /kaggle/input/ is read-only -- attach your chess dataset via the Data tab.

    LOCAL_DIR  = Path("/kaggle/working/datasets")  # writable dataset workspace
    RUNS_DIR   = "/kaggle/working/runs"             # outputs committed with notebook
    INPUT_DIR  = Path("/kaggle/input")              # read-only attached datasets
    DRIVE_DIR  = None                               # no Google Drive on Kaggle

    # Checkpoint: if you attached a previous run output as a Kaggle Dataset,
    # set the path here, e.g. "/kaggle/input/chess-yolo-run2/weights/last.pt".
    # Otherwise the default yolo11m.pt will be downloaded from Ultralytics.
    RESUME_CKPT = "yolo11m.pt"

    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"  LOCAL_DIR  : {LOCAL_DIR}")
    print(f"  INPUT_DIR  : {INPUT_DIR}")
    print(f"  RUNS_DIR   : {RUNS_DIR}")
    print()
    print("[Kaggle] Outputs saved to /kaggle/working/ -- commit via")
    print("         'Save Version' -> 'Save & Run All' to persist them.")

# -- Local -----------------------------------------------------
else:
    _LOCAL_PROJECT = Path(".").resolve()
    LOCAL_DIR  = _LOCAL_PROJECT / "datasets"
    RUNS_DIR   = str(_LOCAL_PROJECT / "runs")
    INPUT_DIR  = None
    DRIVE_DIR  = None
    RESUME_CKPT = "yolo11m.pt"

    LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    print(f"  LOCAL_DIR  : {LOCAL_DIR}")
    print(f"  RUNS_DIR   : {RUNS_DIR}")

print("\nPath configuration complete.")

## Step 1: Install Dependencies
Install all required Python libraries for training.

In [ ]:
# Install dependencies if running in Colab
print("Installing project dependencies...")
# Install extra required libraries for Ultralytics & training
!pip install ultralytics draccus gdown tqdm pillow albumentations pyyaml

## Step 2: Dataset Preparation
To avoid downloading the heavy dataset from the internet every time the runtime restarts, this step will:
1. **Colab**: Check if the dataset already exists on your Google Drive; copy from Drive if found, otherwise download.
2. **Kaggle**: Check if the dataset is attached under `/kaggle/input/` (add it via the Data tab first); symlink or extract directly into the working directory.

In [ ]:
import os
import shutil
import zipfile
import urllib.request
from pathlib import Path
import gdown

# LOCAL_DIR, DRIVE_DIR, INPUT_DIR, ENV are set by Step 0
ANN_URL   = "https://data.4tu.nl/file/99b5c721-280b-450b-b058-b2900b69a90f/3cae6364-daca-4967-b426-1e4b68cdb64c"
ZIP_GD_ID = "1jxmFxjOy0qefdCZ_x3DMNtsvAK4LojEw"

# 1. Prepare annotations.json
local_ann = LOCAL_DIR / "annotations.json"

if not local_ann.exists():
    if ENV == "colab" and DRIVE_DIR is not None:
        # Colab: try Drive backup first
        drive_ann = DRIVE_DIR / "annotations.json"
        if drive_ann.exists():
            print("Copying annotations.json from Google Drive...")
            shutil.copy(drive_ann, local_ann)
        else:
            print("Downloading annotations.json from internet...")
            urllib.request.urlretrieve(ANN_URL, local_ann)
    elif ENV == "kaggle" and INPUT_DIR is not None:
        # Kaggle: search attached read-only input datasets first
        ann_sources = list(INPUT_DIR.glob("**/annotations.json"))
        if ann_sources:
            print(f"Copying annotations.json from Kaggle input: {ann_sources[0]}")
            shutil.copy(ann_sources[0], local_ann)
        else:
            print("Downloading annotations.json from internet...")
            urllib.request.urlretrieve(ANN_URL, local_ann)
    else:
        print("Downloading annotations.json from internet...")
        urllib.request.urlretrieve(ANN_URL, local_ann)
else:
    print("annotations.json already exists.")

# 2. Prepare and extract images
local_images = LOCAL_DIR / "images"
local_zip    = LOCAL_DIR / "preprocessed_images.zip"

if not (local_images.exists() and any(local_images.iterdir())):
    if ENV == "colab" and DRIVE_DIR is not None:
        # Colab: copy zip from Drive then extract to fast local SSD
        drive_zip = DRIVE_DIR / "preprocessed_images.zip"
        if drive_zip.exists():
            print("Copying preprocessed_images.zip from Google Drive...")
            shutil.copy(drive_zip, local_zip)
        else:
            print("Downloading preprocessed_images.zip from internet...")
            gdown.download(id=ZIP_GD_ID, output=str(local_zip), quiet=False)
        print("Extracting images dataset...")
        with zipfile.ZipFile(local_zip, 'r') as zr:
            zr.extractall(LOCAL_DIR)
        local_zip.unlink()  # remove zip to free local SSD space
        print("Extraction completed.")

    elif ENV == "kaggle" and INPUT_DIR is not None:
        # Kaggle: prefer zero-copy symlink to already-extracted images in /kaggle/input/
        img_dirs  = [d for d in INPUT_DIR.glob("**/images") if d.is_dir()]
        zip_files = list(INPUT_DIR.glob("**/preprocessed_images.zip"))
        if img_dirs:
            print(f"Creating symlink for images from Kaggle input: {img_dirs[0]}")
            if local_images.exists():
                if local_images.is_symlink() or local_images.is_file():
                    local_images.unlink()
                else:
                    shutil.rmtree(local_images)
            os.symlink(img_dirs[0], local_images)
        elif zip_files:
            print(f"Extracting preprocessed_images.zip from Kaggle input: {zip_files[0]}")
            with zipfile.ZipFile(zip_files[0], 'r') as zr:
                zr.extractall(LOCAL_DIR)
        else:
            print("Downloading preprocessed_images.zip from internet...")
            gdown.download(id=ZIP_GD_ID, output=str(local_zip), quiet=False)
            print("Extracting images dataset...")
            with zipfile.ZipFile(local_zip, 'r') as zr:
                zr.extractall(LOCAL_DIR)
            local_zip.unlink()
            print("Extraction completed.")
    else:
        # Local fallback: download from internet
        print("Downloading preprocessed_images.zip from internet...")
        gdown.download(id=ZIP_GD_ID, output=str(local_zip), quiet=False)
        print("Extracting images dataset...")
        with zipfile.ZipFile(local_zip, 'r') as zr:
            zr.extractall(LOCAL_DIR)
        local_zip.unlink()
        print("Extraction completed.")
else:
    print("Dataset images are already present and extracted.")

## Step 3: Convert ChessReD COCO dataset format to standard YOLO format
Ultralytics YOLO standard training API works out of the box with the standard YOLODataset format (directory structure of `images/split/` and `labels/split/` along with separate `.txt` labels for each image).
We will run a conversion script to transform ChessReD COCO annotations into standard YOLO text labels, link images into splits without taking extra disk space, and generate a new `dataset.yaml` with correct absolute paths.

In [ ]:
import json
from collections import defaultdict
from pathlib import Path
import numpy as np
from PIL import Image

# Ultralytics imports
from ultralytics.data.dataset import DATASET_CACHE_VERSION, YOLODataset
from ultralytics.data.utils import get_hash, load_dataset_cache_file, save_dataset_cache_file
from ultralytics.models.yolo.detect import DetectionTrainer, DetectionValidator
from ultralytics.utils import TQDM, colorstr

# =====================================================================
# Custom ChessRED Dataset classes for direct in-memory COCO processing
# =====================================================================

class ChessREDDataset(YOLODataset):
    """Custom YOLODataset class that reads ChessRED annotations.json directly in memory.
    """

    def __init__(self, *args, json_file="", split="train", **kwargs):
        """Initialize the ChessRED dataset instance.
        
        Args:
            json_file (str or Path): Path to annotations.json file.
            split (str): Dataset split ('train', 'val', or 'test').
        """
        self.json_file = str(json_file)
        self.split = split
        super().__init__(*args, data={"channels": 3}, **kwargs)

    def get_img_files(self, img_path):
        """Override image path discovery.
        """
        return []

    def cache_labels(self, path=Path("./labels.cache")):
        """Parse ChessRED COCO JSON annotations, normalize bounding boxes, and save to a .cache file."""
        x = {"labels": []}

        if not Path(self.json_file).exists():
            raise FileNotFoundError(f"Annotations JSON file not found: {self.json_file}")

        with open(self.json_file, "r") as f:
            coco = json.load(f)

        # Filter out any non-piece category (such as 'empty') and sort remaining piece categories by id
        valid_cats = [c for c in coco.get("categories", []) if c.get("name", "") != "empty"]
        valid_cats = sorted(valid_cats, key=lambda c: c["id"])

        # Map original category id to 0-indexed contiguous class index
        cat_id_to_cls = {cat["id"]: i for i, cat in enumerate(valid_cats)}

        # Get list of image IDs belonging to current split
        splits = coco.get("splits", {})
        if self.split in splits:
            split_img_ids = set(splits[self.split]["image_ids"])
        else:
            # Fallback: use all image IDs if split is missing or not specified
            split_img_ids = {img["id"] for img in coco.get("images", [])}

        # Group piece annotations by image ID
        img_to_anns = defaultdict(list)
        anns_data = coco.get("annotations", {})
        pieces_list = anns_data.get("pieces", []) if isinstance(anns_data, dict) else (anns_data if isinstance(anns_data, list) else [])

        for ann in pieces_list:
            if ann.get("image_id") in split_img_ids:
                img_to_anns[ann["image_id"]].append(ann)

        # Process each image entry
        for img_info in TQDM(coco.get("images", []), desc=f"Parsing {self.split} annotations"):
            img_id = img_info["id"]
            if img_id not in split_img_ids:
                continue

            h, w = img_info["height"], img_info["width"]
            
            # Resolve image file path relative to json_file parent or img_path
            rel_path = img_info.get("path", img_info.get("file_name", ""))
            json_dir = Path(self.json_file).resolve().parent
            
            im_file = json_dir / rel_path
            if not im_file.exists():
                im_file = Path(self.img_path) / rel_path
            if not im_file.exists():
                im_file = Path(self.img_path).parent / rel_path
            if not im_file.exists() and rel_path.startswith("images/"):
                im_file = Path(self.img_path) / rel_path[len("images/"):]
            if not im_file.exists():
                continue

            # Read actual image dimensions from disk (fast header check via PIL)
            try:
                with Image.open(im_file) as img:
                    actual_w, actual_h = img.size
            except Exception:
                actual_w, actual_h = w, h

            self.im_files.append(str(im_file))
            bboxes = []

            for ann in img_to_anns.get(img_id, []):
                cat_id = ann.get("category_id")
                if cat_id not in cat_id_to_cls:
                    continue

                cls_idx = cat_id_to_cls[cat_id]

                # Bbox format in ChessRED: [x_top_left, y_top_left, width, height] (pixels)
                bbox_raw = ann.get("bbox")
                if bbox_raw is None or len(bbox_raw) != 4:
                    continue

                box = np.array(bbox_raw, dtype=np.float32)
                
                # Convert top-left to center coordinates: [x_center, y_center, width, height]
                box[:2] += box[2:] / 2.0

                # Normalize coordinates by original image dimensions from JSON
                box[[0, 2]] /= w
                box[[1, 3]] /= h

                # Validate bounding box dimensions
                if box[2] <= 0 or box[3] <= 0 or box[0] < 0 or box[1] < 0:
                    continue

                bboxes.append([cls_idx, *box.tolist()])

            lb = np.array(bboxes, dtype=np.float32) if bboxes else np.zeros((0, 5), dtype=np.float32)

            x["labels"].append(
                {
                    "im_file": str(im_file),
                    "shape": (actual_h, actual_w),
                    "cls": lb[:, 0:1],
                    "bboxes": lb[:, 1:],
                    "segments": [],
                    "normalized": True,
                    "bbox_format": "xywh",
                }
            )

        x["hash"] = get_hash([self.json_file, str(self.img_path), self.split])
        save_dataset_cache_file(self.prefix, path, x, DATASET_CACHE_VERSION)
        return x

    def get_labels(self):
        """Retrieve labels from cache file if existing and valid, or trigger parsing."""
        cache_path = Path(self.json_file).parent / f"chessred_{self.split}.cache"
        try:
            cache = load_dataset_cache_file(cache_path)
            assert cache["version"] == DATASET_CACHE_VERSION
            assert cache["hash"] == get_hash([self.json_file, str(self.img_path), self.split])
            self.im_files = [lb["im_file"] for lb in cache["labels"]]
        except (FileNotFoundError, AssertionError, AttributeError, KeyError, ModuleNotFoundError):
            cache = self.cache_labels(cache_path)
        
        cache.pop("hash", None)
        cache.pop("version", None)
        return cache["labels"]


class ChessREDValidator(DetectionValidator):
    """Custom DetectionValidator class integrating ChessREDDataset for validation/evaluation."""

    def build_dataset(self, img_path, mode="val", batch=None):
        """Build and return a ChessREDDataset instance for validation/testing."""
        # Use annotations_json from dataset config, else fall back to LOCAL_DIR set by Step 0
        json_file = self.data.get("annotations_json", str(LOCAL_DIR / "annotations.json"))
        return ChessREDDataset(
            img_path=img_path,
            json_file=json_file,
            split=mode,
            imgsz=self.args.imgsz,
            batch_size=batch,
            augment=False,
            hyp=self.args,
            rect=self.args.rect or mode == "val",
            cache=self.args.cache or None,
            single_cls=self.args.single_cls or False,
            stride=int(self.stride) if hasattr(self, "stride") else 32,
            pad=0.5,
            prefix=colorstr(f"{mode}: "),
            task=self.args.task,
            classes=self.args.classes,
            fraction=1.0,
        )


class ChessREDTrainer(DetectionTrainer):
    """Custom DetectionTrainer class integrating ChessREDDataset into Ultralytics YOLO training loop."""

    def build_dataset(self, img_path, mode="train", batch=None):
        """Build and return a ChessREDDataset instance for the specified mode (train, val, or test)."""
        # Use annotations_json from dataset config, else fall back to LOCAL_DIR set by Step 0
        json_file = self.data.get("annotations_json", str(LOCAL_DIR / "annotations.json"))
        return ChessREDDataset(
            img_path=img_path,
            json_file=json_file,
            split=mode,
            imgsz=self.args.imgsz,
            batch_size=batch,
            augment=mode == "train",
            hyp=self.args,
            rect=self.args.rect or mode == "val",
            cache=self.args.cache or None,
            single_cls=self.args.single_cls or False,
            stride=int(self.model.stride.max()) if hasattr(self, "model") and self.model else 32,
            pad=0.0 if mode == "train" else 0.5,
            prefix=colorstr(f"{mode}: "),
            task=self.args.task,
            classes=self.args.classes,
            fraction=self.args.fraction if mode == "train" else 1.0,
        )

    def get_validator(self):
        """Return a ChessREDValidator instance for validation."""
        self.loss_names = "box_loss", "cls_loss", "dfl_loss"
        return ChessREDValidator(
            self.test_loader, self.save_dir, self.args, getattr(self, "loss_names", None)
        )

## Step 4: Run YOLOv11m Fine-Tuning using Ultralytics API
We load the pretrained YOLOv11m model checkpoint and train it directly using the standard Ultralytics API `YOLO(model).train(...)` pointing to our converted `dataset.yaml` config.

In [ ]:
import torch
from ultralytics import YOLO
from pathlib import Path

# RUNS_DIR and RESUME_CKPT are set by Step 0

# Check GPU availability
device_name = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device_name}")
if device_name == "cuda":
    print(f"GPU Model: {torch.cuda.get_device_name(0)}")

# Initialize YOLO model from checkpoint resolved in Step 0
model = YOLO(RESUME_CKPT)

# Start training using direct Ultralytics API
results = model.train(
    data=str(LOCAL_DIR / "yolo_dataset" / "dataset.yaml"),
    project=RUNS_DIR,
    trainer=ChessREDTrainer,
    epochs=15,
    batch=16,            # Adjust based on GPU VRAM (e.g. 16/32, or -1 for AutoBatch)
    imgsz=1024,
    device=device_name,
    optimizer="auto",
    name="chess_detection_yolo11m",
    resume=False,       # Change to True if you need to resume interrupted training
    exist_ok=False,
)